In [1]:
import tensorflow as tf

import numpy as np
import os
import time

In [2]:
path_to_file = tf.keras.utils.get_file('shakespeare.txt', 'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')

1115394/1115394 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [3]:
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

In [4]:
print(len(text))
print(text[:500])
print(text[-500:])

1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
ldst be: the occasion speaks thee, and
My strong imagination sees a crown
Dropping upon thy head.

SEBASTIAN:
What, art thou waking?

ANTONIO:
Do you not hear me speak?

SEBASTIAN:
I do; and surely
It is a sleepy language and thou speak'st
Out of thy sleep. What is it thou didst say?
This is a strange repose, to be asleep
With eyes wide open; standing, speaking, moving,
And yet so fast asleep.

ANTONIO:
Noble Sebastian,
Thou let'st thy fortune sleep--die, rather; wink'st
Whiles thou art

In [5]:
# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

65 unique characters


In [6]:
text = text.lower()

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer(
    oov_token="<OOV>"
)

tokenizer.fit_on_texts([text])

In [8]:
word_index = tokenizer.word_index

print("Vocabulary size:", len(word_index))

Vocabulary size: 12633


In [9]:
sequences = tokenizer.texts_to_sequences([text])
sequences = sequences[0]
print(sequences[:50])

[89, 270, 140, 36, 970, 144, 669, 128, 16, 103, 34, 103, 103, 89, 270, 7, 41, 34, 1269, 351, 4, 200, 64, 4, 3333, 34, 1269, 1269, 89, 270, 89, 7, 92, 1142, 232, 12, 2275, 581, 4, 2, 306, 34, 36, 2556, 36, 2556, 89, 270, 71, 79]


In [10]:
sequence_length = 20
input_sequences = []

for i in range(sequence_length, len(sequences)):
    input_sequence = sequences[i-sequence_length:i+1]
    input_sequences.append(input_sequence)

In [11]:
import numpy as np

input_sequences = np.array(input_sequences)

print(input_sequences.shape)

(204069, 21)


In [12]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [13]:
print("X:", X[0])
print("y:", y[0])

X: [  89  270  140   36  970  144  669  128   16  103   34  103  103   89
  270    7   41   34 1269  351]
y: 4


In [14]:
print(len(X[0]))

20


In [15]:
vocab_size = len(tokenizer.word_index) + 1
print("Vocabulary size:", vocab_size)

Vocabulary size: 12634


In [16]:
print("Total samples:", len(X))
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Vocabulary size:", vocab_size)

Total samples: 204069
X shape: (204069, 20)
y shape: (204069,)
Vocabulary size: 12634


In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(128),

    Dense(
        vocab_size,
        activation="softmax"
    )
])

In [27]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [28]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 20, 128)        │     1,617,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 12634)          │     1,629,786 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,279,834 (12.51 MB)

 Trainable params: 3,279,834 (12.51 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = model.fit(
    X,
    y,
    epochs=10,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stopping]
)

Epoch 1/10
1276/1276 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.1413 - loss: 5.2486 - val_accuracy: 0.1062 - val_loss: 6.2106
Epoch 2/10
1276/1276 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.1588 - loss: 4.8992 - val_accuracy: 0.1043 - val_loss: 6.3843
Epoch 3/10
1276/1276 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.1792 - loss: 4.5895 - val_accuracy: 0.1009 - val_loss: 6.5662
Epoch 4/10
1276/1276 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2075 - loss: 4.3044 - val_accuracy: 0.0957 - val_loss: 6.7210
Epoch 5/10
1276/1276 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2397 - loss: 4.0397 - val_accuracy: 0.0901 - val_loss: 6.9131
Epoch 6/10
1276/1276 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.2728 - loss: 3.8033 - val_accuracy: 0.0900 - val_loss: 7.0641


In [31]:
model.save("next_word_model.h5")

In [32]:
def my_prediction_function(start_sentence, how_many_words):
    # I'll just make the text lowercase to be safe
    current_text = start_sentence.lower()

    for i in range(how_many_words):
        # Convert words to numbers
        my_list_of_numbers = tokenizer.texts_to_sequences([current_text])[0]

        # Just keep the last few numbers so the model doesn't get confused
        # I'll use sequence_length which is 20
        if len(my_list_of_numbers) > 20:
            my_list_of_numbers = my_list_of_numbers[-20:]

        # The model needs a specific shape, so I'll add an extra dimension
        input_for_model = np.array([my_list_of_numbers])

        # Make the prediction
        all_predictions = model.predict(input_for_model, verbose=0)

        # Find which number has the highest score
        best_number = 0
        highest_score = -1
        for j in range(len(all_predictions[0])):
            if all_predictions[0][j] > highest_score:
                highest_score = all_predictions[0][j]
                best_number = j

        # Look through the dictionary to find the word for that number
        the_word_we_found = ""
        for word, index in tokenizer.word_index.items():
            if index == best_number:
                the_word_we_found = word
                break

        # Add the word to the sentence
        current_text = current_text + " " + the_word_we_found

    return current_text

# Run it and see what happens
result = my_prediction_function("First Citizen: You are all", 10)
print(result)

first citizen: you are all the volsces have you have been a very man to


In [33]:
# Try one more time with another sentence
result2 = my_prediction_function("to be or not to be", 10)
print(result2)

to be or not to be a king of mine own hands the king's name is
